In [19]:

import os, sys
import pandas as pd
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
sys.path.append(os.path.abspath(".."))
from src.semantic import semantic_retriever_rag, load_semantic
from src.rag_pipeline import build_context


In [9]:
load_dotenv()
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [10]:


llm_endpoint = HuggingFaceEndpoint(
    repo_id = "meta-llama/Meta-Llama-3-8B-Instruct",
    task = "text-generation",
    max_new_tokens = 200,
    huggingfacehub_api_token = hf_token,
)

llm = ChatHuggingFace(llm=llm_endpoint)

In [11]:
test = llm.invoke("Recommend a beginner book about investing in one short paragraph.")
print(test.content)

I recommend "A Random Walk Down Wall Street" by Burton G. Malkiel as a beginner book about investing. First published in 1973, this book provides a comprehensive and accessible introduction to investing and the stock market. Malkiel, a renowned expert, explains complex concepts in simple terms, covering topics such as asset allocation, diversification, and the myths and realities of investing. This book is perfect for beginners who want to understand the basics of investing and make informed decisions about their money.


In [17]:
merged_df = pd.read_parquet("../data/processed/merged.parquet")
semantic_model, semantic_index, semantic_meta = load_semantic()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7024.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
test_docs = semantic_retriever_rag(
    query="good books for beginners learning investing",
    model=semantic_model,
    index=semantic_index,
    df=merged_df,
    top_k=3,
)

test_docs.head()

,parent_asin,product_title,review_title,review_text,rating,semantic_score
3496,B01AR5417A,Investment: A History (Columbia Business Schoo...,Long and tiresome,The book is long and tiresome. I didn't learn ...,3.0,0.491082
3021,B008Y1L7EA,Little Kids Big Money: Tools for Teaching Kid ...,Must Read For Parents,This book really makes a lot of sense on how t...,5.0,0.444927
3609,B005T75AF4,The Node Beginner Book,A good resource for starting Node,"A good resource for starting Node, but it fall...",3.0,0.438229


In [20]:

test_context = build_context(test_docs)
print(test_context)

Product ASIN: B01AR5417A
Title: Investment: A History (Columbia Business School Publishing)
Review Title: Long and tiresome
Rating: 3.0
Review Text: The book is long and tiresome. I didn't learn any interesting theory or facts. The main thesis is that nowadays more people invest while in old ages only the elite invested. But this something that I knew (and I believe everybody else knew) before reading this book.

---

Product ASIN: B008Y1L7EA
Title: Little Kids Big Money: Tools for Teaching Kid Friendly Finance
Review Title: Must Read For Parents
Rating: 5.0
Review Text: This book really makes a lot of sense on how to teach a child about money, how to budget money, and how to really think about money. I love how they talk about different incentives for different types of children. Yes, this author realizes that not all children are the same and what works for one, doesn't work for another.

---

Product ASIN: B005T75AF4
Title: The Node Beginner Book
Review Title: A good resource for st